# RAG Pipeline — Core Build Report

**Notebook:** `rag_pipeline.ipynb`  |  **Stack:** `pypdf` → fixed-size chunking → `sentence-transformers` → `chromadb` (persistent) → `ollama` (local LLM)

---

## Executive Summary

This notebook builds the **core Retrieval-Augmented Generation (RAG) pipeline** for a document Q&A application. It is written as a reproducible report: every stage has a short design rationale (Markdown), the implementation (modular, commented Python), and a verification step.

| Stage | Section | Output |
|---|---|---|
| Setup | 1 | Imports, a single `RAGConfig`, environment checks |
| Load & Inspect | 2.1 | Clean page/section records + failure log |
| Chunking | 2.2 | Fixed-size, overlapping chunks with source metadata |
| Embeddings & Vector Store | 2.3 | Persistent ChromaDB collection on disk |
| Retrieval & Prompting | 2.4 | Top-k retriever, grounded prompt, local Ollama LLM, `[S#]` citations |
| Evaluation | 2.6 | 13-question results table + failure analysis |
| Export | 2.7 | `data/vector_store/` (vector DB + `config.json`) ready for a FastAPI backend |

### Architecture

```
 PDF / TXT / MD files
        │
        ▼
 Loader (pypdf / text)  ──►  failure log (never crashes the run)
        │
        ▼
 Chunker (fixed size + overlap, whitespace-aware)
        │
        ▼
 Embedder (sentence-transformers, normalized vectors)
        │
        ▼
 ChromaDB PersistentClient  ◄──── saved to data/vector_store/  (+ config.json)
        │
        ▼
 Retriever (top-k cosine) ──► similarity gate (abstain if nothing relevant)
        │
        ▼
 Prompt (numbered context + citation rules) ──► Ollama LLM ──► answer with [S#] citations
        │
        ▼
 Evaluation (13 questions) ──► results table ──► export
```

> **Numbering note:** section numbers follow the project spec exactly (2.1, 2.2, 2.3, 2.4, 2.6, 2.7).

---
## 1. Prerequisites & Setup

**System requirements**

- Python **3.10+**
- [Ollama](https://ollama.com) installed and running locally (`ollama serve`, started automatically by the desktop app)
- A pulled model — the default here is a small, fast one: `ollama pull llama3.2`
- ~1 GB free disk for the embedding model download (first run only)

**Design decision — one configuration object.** Every tunable (paths, chunk size, embedding model, `top_k`, LLM name, …) lives in a single `RAGConfig` dataclass. That object is later serialized to `data/vector_store/config.json` (section 2.7), so the FastAPI backend reads *exactly* the settings the index was built with. A mismatch between the embedding model used at index time and query time is one of the most common silent RAG bugs; this design removes it.

In [1]:
# Run once (uncomment). In production, pin versions in requirements.txt.
# %pip install -q pandas numpy chromadb sentence-transformers pypdf ollama

In [2]:
# ---- Standard library ----
import os
os.environ.setdefault("ANONYMIZED_TELEMETRY", "False")      # stop Chroma telemetry calls (can stall/pollute output when offline)
os.environ.setdefault("TOKENIZERS_PARALLELISM", "false")     # avoid tokenizer fork warnings/deadlocks in notebooks
os.environ.setdefault("HF_HUB_DISABLE_PROGRESS_BARS", "1")   # no HF download bars: in VS Code they need ipywidgets files from a CDN and can freeze the UI
os.environ.setdefault("TQDM_DISABLE", "1")                    # plain-text/no tqdm bars everywhere (avoids the "Widgets require us to download..." popup)
import json
import re
import sys
import time
import hashlib
import logging
import subprocess
import warnings
from dataclasses import dataclass, asdict
from datetime import datetime, timezone
from importlib.metadata import version, PackageNotFoundError
from pathlib import Path
from typing import Dict, List, Optional, Tuple

# ---- Third-party ----
import numpy as np
import pandas as pd
import chromadb                                   # persistent vector database
import ollama                                     # client for the local LLM server
from pypdf import PdfReader                       # PDF text extraction
from sentence_transformers import SentenceTransformer   # embedding model
from IPython.display import Markdown, display

# ---- Logging: INFO for our code, quiet for noisy libraries ----
logging.basicConfig(level=logging.INFO, format="%(asctime)s | %(levelname)s | %(message)s")
logger = logging.getLogger("rag_pipeline")
logging.getLogger("pypdf").setLevel(logging.ERROR)     # pypdf warns loudly on slightly malformed PDFs
for noisy in ("httpx", "httpcore", "huggingface_hub", "sentence_transformers", "urllib3", "chromadb"):
    logging.getLogger(noisy).setLevel(logging.WARNING)   # hide the per-request "HTTP Request: HEAD ..." spam
warnings.filterwarnings("ignore", category=FutureWarning)

np.random.seed(42)                                     # reproducibility for anything stochastic

# ---- Record library versions (useful in bug reports and for the backend team) ----
for pkg in ["pandas", "numpy", "chromadb", "sentence-transformers", "pypdf", "ollama"]:
    try:
        print(f"{pkg:<24}{version(pkg)}")
    except PackageNotFoundError:
        print(f"{pkg:<24}NOT INSTALLED  ->  pip install {pkg}")

pandas                  2.3.3
numpy                   2.3.3
chromadb                1.5.9
sentence-transformers   6.1.0
pypdf                   6.19.0
ollama                  0.6.2


In [3]:
@dataclass
class RAGConfig:
    """Single source of truth for the whole pipeline. Serialized to config.json in section 2.7."""

    # --- Paths (relative to the notebook's working directory) ---
    data_dir: str = "D:/ITI Computer Vision/Final Project/data/documents"              # where the source PDFs / text files live
    vector_store_dir: str = "D:/ITI Computer Vision/Final Project/data/vector_store"   # persisted index + config (consumed by FastAPI)

    # --- Chunking (characters, not tokens; see the justification in section 2.2) ---
    chunk_size: int = 800
    chunk_overlap: int = 150

    # --- Embeddings ---
    embedding_model: str = "sentence-transformers/all-MiniLM-L6-v2"   # 384-dim, fast, CPU-friendly
    embedding_batch_size: int = 64

    # --- Vector store ---
    collection_name: str = "rag_chunks"
    distance_metric: str = "cosine"               # cosine works best with normalized sentence embeddings

    # --- Retrieval & generation ---
    top_k: int = 5                                # how many chunks go into the prompt
    min_similarity: float = 0.25                  # abstain (skip the LLM) if the best hit is weaker than this
    ollama_model: str = "llama3.2"                # any model you have pulled: `ollama list`
    temperature: float = 0.0                      # deterministic output => reproducible evaluation
    num_ctx: int = 4096                           # context window requested from Ollama

    # --- Loader ---
    supported_extensions: tuple = (".pdf", ".txt", ".md")


CFG = RAGConfig()

# Sanity checks: fail fast on impossible settings.
assert CFG.chunk_overlap < CFG.chunk_size, "chunk_overlap must be smaller than chunk_size"
assert CFG.top_k >= 1

Path(CFG.data_dir).mkdir(parents=True, exist_ok=True)
Path(CFG.vector_store_dir).mkdir(parents=True, exist_ok=True)

print(json.dumps(asdict(CFG), indent=2))

{
  "data_dir": "D:/ITI Computer Vision/Final Project/data/documents",
  "vector_store_dir": "D:/ITI Computer Vision/Final Project/data/vector_store",
  "chunk_size": 800,
  "chunk_overlap": 150,
  "embedding_model": "sentence-transformers/all-MiniLM-L6-v2",
  "embedding_batch_size": 64,
  "collection_name": "rag_chunks",
  "distance_metric": "cosine",
  "top_k": 5,
  "min_similarity": 0.25,
  "ollama_model": "llama3.2",
  "temperature": 0.0,
  "num_ctx": 4096,
  "supported_extensions": [
    ".pdf",
    ".txt",
    ".md"
  ]
}


---
## 2. Pipeline Build

### 2.1 Load & Inspect

**Goal:** turn a directory of PDFs and text files into clean, uniformly structured records, and know *exactly* what did and did not load.

**Approach**

- Walk `data/documents/` recursively; accept `.pdf`, `.txt`, `.md`.
- **PDFs** are read page by page with `pypdf`, so every record carries a **page number** (needed for precise citations later).
- **Text/Markdown** files are read as UTF-8 with a Latin-1 fallback, so a stray encoding never kills the run.
- Every file is wrapped in its own `try/except`: **one bad file must never abort the batch.** Failures are logged and collected in a table for review.

### Domain and data

**Domain:** the internal knowledge base of a fictional robotics company, *Helios Robotics* (warehouse robot **HX-200**). The corpus has **10 documents in 3 formats** (5 PDF, 2 Markdown, 3 plain text) and about 8,600 words:

| Area | Documents |
|---|---|
| HR & finance | `hr_leave_policy.pdf`, `expense_policy.pdf`, `onboarding_guide.txt`, `remote_work_policy.txt` |
| Product | `hx200_manual.pdf`, `warranty_and_support.pdf`, `customer_faq.txt` |
| Engineering & security | `engineering_handbook.md`, `incident_runbook.md`, `security_policy.pdf` |

**Important: the corpus is synthetic.** The documents were written for this project; the company and every number in them are invented. That is deliberate: an LLM cannot answer from its own training data, so any correct answer must come from retrieval, and any invented answer is a visible hallucination. It also lets us test refusal (questions the corpus cannot answer).

**Built-in difficulty.** Several numbers and phrases recur across documents on purpose (e.g. "within 1 hour" appears in the security policy, the incident runbook and the warranty terms; "500" is both an inspection interval and an expense approval limit). PDF pages also carry a header/footer line that the text extractor includes. These are realistic sources of retrieval mix-ups that the evaluation in 2.6 can surface.

**Where the files live:** `data/documents/` (path set in `RAGConfig`).

In [4]:
# Verify the corpus folder is populated before loading anything.
corpus_files = sorted(p for p in Path(CFG.data_dir).rglob("*") if p.is_file())

if not corpus_files:
    raise FileNotFoundError(
        f"No documents found in {Path(CFG.data_dir).resolve()}. "
        "Copy the 10 corpus files (PDF / MD / TXT) into that folder and re-run."
    )

print(f"{len(corpus_files)} file(s) in {CFG.data_dir}:")
for p in corpus_files:
    print(f"  {p.name:<30}{p.stat().st_size / 1024:>8.1f} KB")

10 file(s) in D:/ITI Computer Vision/Final Project/data/documents:
  customer_faq.txt                   3.9 KB
  engineering_handbook.md            4.8 KB
  expense_policy.pdf                 7.1 KB
  hr_leave_policy.pdf                7.8 KB
  hx200_manual.pdf                   7.6 KB
  incident_runbook.md                4.8 KB
  onboarding_guide.txt               3.5 KB
  remote_work_policy.txt             4.3 KB
  security_policy.pdf                6.9 KB
  warranty_and_support.pdf           6.5 KB


In [5]:
def read_pdf_pages(path: Path) -> List[Dict]:
    """Extract text page by page. Raises ValueError if nothing extractable (e.g. scanned PDF)."""
    reader = PdfReader(str(path))

    # Some PDFs are "encrypted" only to restrict editing; the empty password opens them.
    if reader.is_encrypted:
        if reader.decrypt("") == 0:            # 0 == decryption failed
            raise ValueError("PDF is password-protected")

    pages = []
    for page_no, page in enumerate(reader.pages, start=1):
        text = page.extract_text() or ""
        if text.strip():                        # skip blank / image-only pages
            pages.append({"page": page_no, "text": text})

    if not pages:
        raise ValueError("no extractable text (scanned/image-only PDF? OCR would be required)")
    return pages


def read_text_file(path: Path) -> str:
    """Read a text file, trying UTF-8 first and falling back to Latin-1 (which never fails)."""
    try:
        return path.read_text(encoding="utf-8")
    except UnicodeDecodeError:
        logger.warning("%s is not valid UTF-8; falling back to Latin-1", path.name)
        return path.read_text(encoding="latin-1")


def load_documents(directory: str, cfg: RAGConfig) -> Tuple[List[Dict], List[Dict], List[Dict]]:
    """
    Recursively load every supported file under `directory`.

    Returns
    -------
    docs     : one record per PDF page, or per text file
    failures : files that were supported but could not be parsed (with the error)
    skipped  : files ignored because of an unsupported extension
    """
    root = Path(directory)
    if not root.exists():
        raise FileNotFoundError(f"Document directory not found: {root.resolve()}")

    docs, failures, skipped = [], [], []
    for path in sorted(root.rglob("*")):
        if not path.is_file() or path.name.startswith("."):
            continue
        ext = path.suffix.lower()
        rel = path.relative_to(root).as_posix()      # stable, readable identifier used in citations

        if ext not in cfg.supported_extensions:
            skipped.append({"file": rel, "reason": f"unsupported format '{ext or 'none'}'"})
            continue

        try:
            if ext == ".pdf":
                for p in read_pdf_pages(path):
                    docs.append({"source": rel, "format": "pdf", "page": p["page"], "text": p["text"]})
            else:
                text = read_text_file(path)
                if not text.strip():
                    raise ValueError("file is empty")
                docs.append({"source": rel, "format": ext.lstrip("."), "page": 0, "text": text})  # page 0 == n/a
        except Exception as exc:                      # deliberately broad: one bad file must not stop the batch
            failures.append({"file": rel, "format": ext.lstrip("."), "error": f"{type(exc).__name__}: {exc}"})
            logger.warning("FAILED to parse %s -> %s", rel, exc)

    return docs, failures, skipped

In [6]:
docs, failures, skipped = load_documents(CFG.data_dir, CFG)

if not docs:
    raise RuntimeError(f"No documents could be loaded from {CFG.data_dir}. Add PDF/TXT/MD files and re-run.")

# ---- Inspection tables ----
docs_df = pd.DataFrame(docs)
docs_df["n_chars"] = docs_df["text"].str.len()
docs_df["n_words"] = docs_df["text"].str.split().str.len()

files_df = (
    docs_df.groupby(["source", "format"], as_index=False)
    .agg(records=("page", "count"), chars=("n_chars", "sum"), words=("n_words", "sum"))
)
display(files_df)

if failures:
    print("\nFiles that failed to parse:")
    display(pd.DataFrame(failures))
if skipped:
    print("\nFiles skipped (unsupported):")
    display(pd.DataFrame(skipped))

# ---- Auto-generated summary (numbers are computed, so it is always accurate) ----
fmt_counts = files_df["format"].value_counts().to_dict()
n_files = files_df["source"].nunique()
summary_md = f"""
#### 2.1 Summary (generated from this run)

- **Documents loaded:** {n_files} file(s) → {len(docs_df)} page/section record(s), {docs_df['n_words'].sum():,} words in total.
- **Formats:** {', '.join(f'{k.upper()}: {v}' for k, v in fmt_counts.items())}.
- **Failed parses:** {len(failures)} file(s){' — ' + ', '.join(f['file'] for f in failures) if failures else ''}.
- **Skipped (unsupported extension):** {len(skipped)} file(s).
"""
display(Markdown(summary_md))

,source,format,records,chars,words
0,customer_faq.txt,txt,1,4022,684
1,engineering_handbook.md,md,1,4902,819
2,expense_policy.pdf,pdf,3,6095,1009
3,hr_leave_policy.pdf,pdf,3,7114,1220
4,hx200_manual.pdf,pdf,3,6390,1088
5,incident_runbook.md,md,1,4869,785
6,onboarding_guide.txt,txt,1,3569,600
7,remote_work_policy.txt,txt,1,4384,709
8,security_policy.pdf,pdf,3,5678,882
9,warranty_and_support.pdf,pdf,3,5074,819



#### 2.1 Summary (generated from this run)

- **Documents loaded:** 10 file(s) → 20 page/section record(s), 8,615 words in total.
- **Formats:** PDF: 5, TXT: 3, MD: 2.
- **Failed parses:** 0 file(s).
- **Skipped (unsupported extension):** 0 file(s).


#### 2.1 Summary — how the loader treats each case

| Situation | Handling |
|---|---|
| **Supported files** (`.pdf`, `.txt`, `.md`) | Loaded into records `{source, format, page, text}`. PDFs yield one record per non-empty page; text files yield one record. |
| **Corrupt / malformed PDF** | `pypdf` raises → caught per file → logged in the failure table with the exception type and message. The rest of the corpus still loads. |
| **Scanned (image-only) PDF** | No text layer, so extraction returns nothing. Reported as *"no extractable text"* instead of silently indexing an empty document. Fix: run OCR (e.g. `ocrmypdf`) and drop the result in the folder. |
| **Password-protected PDF** | The empty password is tried first (common for edit-locked files); otherwise reported as a failure. |
| **Non-UTF-8 text** | Falls back to Latin-1, with a warning in the log. |
| **Empty files** | Reported as failures (an empty document would only add noise to the index). |
| **Unsupported extensions** (`.docx`, `.png`, …) | Listed as *skipped*, never silently dropped. |

The exact document count, format breakdown, and failure list for this run are printed by the cell above (*"2.1 Summary (generated from this run)"*).

---
### 2.2 Chunking Strategy

**Goal:** split each record into fixed-size, overlapping windows. Chunks are the unit of retrieval: too big and the embedding blurs several topics; too small and the answer is cut off from its context.

**Implementation choices**

- **Fixed-size windows by characters** (`chunk_size=800`, `chunk_overlap=150`) — simple, deterministic, and easy to reason about.
- **Whitespace-aware boundaries:** the window end is pulled back to the nearest space (within a small look-back), and the next start is moved forward to a word boundary, so words are never cut in half.
- **PDFs are chunked page by page**, so every chunk keeps an exact page number for citations.
- **Every chunk carries metadata** (`source`, `page`, `chunk_index`, `char_start`, `char_end`) and a **stable ID** (`<source>::<page>::chunk<NNN>`), which makes citations, debugging, and idempotent re-indexing straightforward.

In [7]:
def normalize_whitespace(text: str) -> str:
    """Collapse runs of whitespace/newlines. PDF extraction often yields ragged line breaks."""
    return re.sub(r"\s+", " ", text).strip()


def chunk_text(text: str, chunk_size: int, overlap: int) -> List[Tuple[int, int, str]]:
    """
    Split `text` into fixed-size overlapping chunks, snapping boundaries to whitespace.

    Returns a list of (start_char, end_char, chunk_text). Offsets refer to the
    whitespace-normalized text, not the raw file.
    """
    if overlap >= chunk_size:
        raise ValueError("overlap must be smaller than chunk_size (otherwise the window never advances)")

    text = normalize_whitespace(text)
    n = len(text)
    if n == 0:
        return []
    if n <= chunk_size:                               # short text -> a single chunk
        return [(0, n, text)]

    lookback = int(chunk_size * 0.15)                 # how far we may retreat to find a space
    chunks: List[Tuple[int, int, str]] = []
    start = 0

    while start < n:
        end = min(start + chunk_size, n)

        # Pull the end back to the last space in the look-back window (avoid cutting mid-word).
        if end < n:
            space = text.rfind(" ", max(start + 1, end - lookback), end)
            if space != -1:
                end = space

        piece = text[start:end].strip()
        if piece:
            chunks.append((start, end, piece))

        if end >= n:                                  # reached the end of the text
            break

        # Next window starts `overlap` characters before this one ended ...
        next_start = end - overlap
        # ... nudged forward to a word boundary so we don't begin mid-word.
        if next_start > 0 and text[next_start - 1] != " ":
            space = text.find(" ", next_start, end)
            if space != -1:
                next_start = space + 1

        start = max(next_start, start + 1)            # guarantee forward progress (no infinite loop)

    return chunks


def chunk_documents(docs: List[Dict], cfg: RAGConfig) -> List[Dict]:
    """Chunk every loaded record and attach metadata + a stable chunk ID."""
    all_chunks: List[Dict] = []
    counters: Dict[Tuple[str, int], int] = {}         # chunk index restarts for each (source, page)

    for doc in docs:
        key = (doc["source"], doc["page"])
        for start, end, piece in chunk_text(doc["text"], cfg.chunk_size, cfg.chunk_overlap):
            idx = counters.get(key, 0)
            counters[key] = idx + 1
            loc = f"p{doc['page']}" if doc["page"] else "full"
            all_chunks.append({
                "chunk_id": f"{doc['source']}::{loc}::chunk{idx:03d}",
                "text": piece,
                "source": doc["source"],
                "format": doc["format"],
                "page": doc["page"],                  # 0 means "not applicable" (text files)
                "chunk_index": idx,
                "char_start": start,
                "char_end": end,
            })
    return all_chunks

In [8]:
chunks = chunk_documents(docs, CFG)
chunks_df = pd.DataFrame(chunks)
chunks_df["n_chars"] = chunks_df["text"].str.len()

assert chunks_df["chunk_id"].is_unique, "chunk IDs must be unique (Chroma upserts by ID)"

print(f"{len(docs)} records  →  {len(chunks)} chunks  (chunk_size={CFG.chunk_size}, overlap={CFG.chunk_overlap})\n")
display(chunks_df["n_chars"].describe().round(1).to_frame("chunk length (chars)").T)
display(chunks_df.groupby("source").size().rename("n_chunks").to_frame())

# Show the overlap in action: the tail of chunk 0 should reappear at the head of chunk 1.
multi = chunks_df.groupby(["source", "page"]).filter(lambda g: len(g) >= 2)   # records that produced 2+ chunks
if not multi.empty:
    first_source = multi["source"].iloc[0]
    pair = multi[multi["source"] == first_source].head(2)
    print(f"\nOverlap check for {first_source}:")
    print("  end of chunk 0  :", "…" + pair.iloc[0]["text"][-80:])
    print("  start of chunk 1:", pair.iloc[1]["text"][:80] + "…")
else:
    print("\n(No document was long enough to need more than one chunk.)")

20 records  →  86 chunks  (chunk_size=800, overlap=150)



,count,mean,std,min,25%,50%,75%,max
chunk length (chars),86.0,717.1,167.2,220.0,791.2,795.0,797.0,799.0


,n_chunks
source,
customer_faq.txt,6
engineering_handbook.md,8
expense_policy.pdf,10
hr_leave_policy.pdf,12
hx200_manual.pdf,11
incident_runbook.md,8
onboarding_guide.txt,6
remote_work_policy.txt,7
security_policy.pdf,9



Overlap check for customer_faq.txt:
  end of chunk 0  : … does the battery last? About 8 hours per charge. A full charge takes 2.5 hours,
  start of chunk 1: is intended for temperatures from -25 to 10 degrees Celsius. How long does the b…


#### 2.2 Justification of chunk size and overlap

**Chunk size = 800 characters (≈ 150–200 English tokens).**
1. *It fits the embedding model.* `all-MiniLM-L6-v2` truncates its input at **256 word-piece tokens**; text beyond that is silently ignored by the encoder. 800 characters leaves comfortable headroom (verified with a real tokenizer in section 2.3), so no chunk loses its tail.
2. *Precision vs. context.* A chunk of roughly one to three paragraphs usually covers a single topic, giving a sharp embedding that discriminates well under cosine similarity. Much smaller chunks (<300 chars) separate facts from the context that gives them meaning (a number from its subject); much larger ones dilute the vector with unrelated content.
3. *Prompt budget.* With `top_k=4`, the context is about 3,200 characters (≈ 800 tokens). That fits easily inside a 4,096-token window alongside the instructions and the answer, which matters for small local models that degrade when the prompt is crowded.

**Overlap = 150 characters (~19% of the chunk).**
1. *Boundary protection.* A fact that straddles a boundary (e.g. a sentence that starts at character 790) is otherwise split between two chunks and may be retrievable from neither. The overlap guarantees it appears whole in at least one chunk. The common guideline is 10–20%.
2. *Bounded cost.* The step size is 650 characters, so the index is only ≈ 1.23× larger than with no overlap. Larger overlaps (>25%) inflate the index and return near-duplicate hits that waste `top_k` slots.

**Boundary handling and known trade-offs.** Windows are snapped to whitespace so words are never cut. PDFs are chunked per page, which gives exact page citations but means a paragraph that continues across a page break is not merged. **When to revisit:** for very fact-dense text (FAQs, tables) try ~400/60; for narrative text or a longer-context embedding model (e.g. 512-token BGE models) try ~1,200/200, and re-run the evaluation in 2.6 to compare.

---
### 2.3 Embeddings & Vector Store

**Goal:** embed every chunk, store vectors + text + metadata in **ChromaDB**, and **persist to disk** so the index can be re-opened later without recomputing anything.

**Key design points**

- **Normalized embeddings + cosine distance.** Sentence-transformer vectors are L2-normalized, so cosine similarity is well defined and `similarity = 1 − distance` in Chroma.
- **We supply the embeddings ourselves** (not Chroma's default embedder), so the model is explicit and recorded in `config.json`.
- **`PersistentClient`** writes to `data/vector_store/chroma/`. Nothing else is required to reload it.
- **Build-or-load logic.** A fingerprint (SHA-256 over all chunk IDs + texts) and the config values are stored next to the index. On re-runs, if nothing changed, the expensive embedding step is **skipped**; if the corpus or settings changed, the store is rebuilt so it can never be stale.
- **Idempotent writes** via `upsert` in batches.

In [9]:
# Load the embedding model (downloads ~90 MB the first time, then cached by Hugging Face).
t0 = time.time()
embedder = SentenceTransformer(CFG.embedding_model)
print(f"Loaded {CFG.embedding_model} in {time.time() - t0:.1f}s  |  max_seq_length = {embedder.max_seq_length} tokens")

# --- Verify the chunk-size choice from 2.2: how many chunks exceed the encoder's window? ---
token_ids = embedder.tokenizer([c["text"] for c in chunks], add_special_tokens=True, truncation=False)["input_ids"]
tok_lens = pd.Series([len(ids) for ids in token_ids], name="tokens")
over = int((tok_lens > embedder.max_seq_length).sum())
print(f"Chunk token lengths -> mean {tok_lens.mean():.0f}, max {tok_lens.max()}; "
      f"{over} of {len(tok_lens)} chunks exceed the {embedder.max_seq_length}-token limit")
if over:
    print("⚠️  Some chunks will be truncated by the encoder. Reduce chunk_size in RAGConfig.")

2026-09-21 19:43:51,896 | WARNING | Warning: You are sending unauthenticated requests to the HF Hub. Please set a HF_TOKEN to enable higher rate limits and faster downloads.


Loaded sentence-transformers/all-MiniLM-L6-v2 in 6.5s  |  max_seq_length = 256 tokens
Chunk token lengths -> mean 156, max 202; 0 of 86 chunks exceed the 256-token limit


In [10]:
def corpus_fingerprint(chunks: List[Dict]) -> str:
    """Hash of all chunk IDs + texts. Changes whenever the corpus or the chunking changes."""
    h = hashlib.sha256()
    for c in chunks:
        h.update(c["chunk_id"].encode("utf-8"))
        h.update(c["text"].encode("utf-8"))
    return h.hexdigest()


def chroma_path(cfg: RAGConfig) -> Path:
    return Path(cfg.vector_store_dir) / "chroma"


def config_path(cfg: RAGConfig) -> Path:
    return Path(cfg.vector_store_dir) / "config.json"


def save_store_config(cfg: RAGConfig, updates: Dict) -> Dict:
    """Merge the RAGConfig + extra build info into config.json (existing keys are preserved)."""
    path = config_path(cfg)
    path.parent.mkdir(parents=True, exist_ok=True)
    current = json.loads(path.read_text(encoding="utf-8")) if path.exists() else {}
    current.update(asdict(cfg))
    current.update(updates)
    path.write_text(json.dumps(current, indent=2, ensure_ascii=False), encoding="utf-8")
    return current


def embed_chunks(chunks: List[Dict], model: SentenceTransformer, cfg: RAGConfig) -> np.ndarray:
    """Embed all chunk texts in batches. Returns a float32 array of shape (n_chunks, dim)."""
    return model.encode(
        [c["text"] for c in chunks],
        batch_size=cfg.embedding_batch_size,
        show_progress_bar=False,            # progress widgets can freeze VS Code notebooks; the corpus is small anyway
        normalize_embeddings=True,          # unit-length vectors => cosine similarity
        convert_to_numpy=True,
    )


def build_vector_store(chunks: List[Dict], embeddings: np.ndarray, cfg: RAGConfig):
    """(Re)create the Chroma collection on disk and fill it with chunks + embeddings + metadata."""
    client = chromadb.PersistentClient(path=str(chroma_path(cfg)))

    # Start from a clean collection so removed documents don't linger in the index.
    try:
        client.delete_collection(cfg.collection_name)
    except Exception:                        # exception type differs across Chroma versions; "not found" is fine
        pass

    collection = client.get_or_create_collection(
        name=cfg.collection_name,
        metadata={"hnsw:space": cfg.distance_metric},   # distance function for the HNSW index
    )

    batch = 500                              # stay well below Chroma's max batch size
    for i in range(0, len(chunks), batch):
        part = chunks[i:i + batch]
        collection.upsert(
            ids=[c["chunk_id"] for c in part],
            embeddings=embeddings[i:i + batch].tolist(),
            documents=[c["text"] for c in part],
            # Chroma metadata values must be str/int/float/bool (no None) -> page uses 0 for "n/a".
            metadatas=[{k: c[k] for k in ("source", "format", "page", "chunk_index", "char_start", "char_end")}
                       for c in part],
        )
    return collection


def store_is_current(cfg: RAGConfig, fingerprint: str) -> bool:
    """True if the on-disk store was built from the same corpus AND the same settings."""
    path = config_path(cfg)
    if not path.exists():
        return False
    saved = json.loads(path.read_text(encoding="utf-8"))
    keys = ("chunk_size", "chunk_overlap", "embedding_model", "collection_name", "distance_metric")
    if any(saved.get(k) != getattr(cfg, k) for k in keys) or saved.get("corpus_fingerprint") != fingerprint:
        return False
    try:
        col = chromadb.PersistentClient(path=str(chroma_path(cfg))).get_collection(cfg.collection_name)
        return col.count() == saved.get("num_chunks")
    except Exception:
        return False


def load_vector_store(vector_store_dir: str, embedder: Optional[SentenceTransformer] = None):
    """
    Re-open a persisted store WITHOUT rebuilding anything.
    This is exactly what the FastAPI backend does at startup.
    Returns (collection, embedder, saved_config).
    """
    vs = Path(vector_store_dir)
    saved = json.loads((vs / "config.json").read_text(encoding="utf-8"))
    client = chromadb.PersistentClient(path=str(vs / "chroma"))
    collection = client.get_collection(saved["collection_name"])
    if embedder is None:                     # the backend loads the model named in the config
        embedder = SentenceTransformer(saved["embedding_model"])
    return collection, embedder, saved

In [11]:
FORCE_REBUILD = False        # set True to ignore the on-disk store and rebuild from scratch

fingerprint = corpus_fingerprint(chunks)

if not FORCE_REBUILD and store_is_current(CFG, fingerprint):
    logger.info("On-disk vector store matches the current corpus and settings -> skipping embedding.")
else:
    logger.info("Building vector store (%d chunks)...", len(chunks))
    embeddings = embed_chunks(chunks, embedder, CFG)
    build_vector_store(chunks, embeddings, CFG)
    save_store_config(CFG, {
        "embedding_dimension": int(embeddings.shape[1]),
        "num_chunks": len(chunks),
        "num_documents": int(files_df["source"].nunique()),
        "corpus_fingerprint": fingerprint,
        "built_at_utc": datetime.now(timezone.utc).isoformat(timespec="seconds"),
    })
    logger.info("Vector store persisted to %s", Path(CFG.vector_store_dir).resolve())

2026-09-21 19:43:59,193 | INFO | On-disk vector store matches the current corpus and settings -> skipping embedding.


In [12]:
# ---- Prove persistence: re-open the store from disk like a fresh process would. ----
collection, embedder, saved_cfg = load_vector_store(CFG.vector_store_dir, embedder=embedder)

print("Collection        :", collection.name)
print("Chunks in store   :", collection.count(), "(expected:", saved_cfg["num_chunks"], ")")
print("Embedding model   :", saved_cfg["embedding_model"], f"({saved_cfg['embedding_dimension']} dims)")
print("Chunk size/overlap:", saved_cfg["chunk_size"], "/", saved_cfg["chunk_overlap"])
assert collection.count() == saved_cfg["num_chunks"], "Persisted store does not match the config"

Collection        : rag_chunks
Chunks in store   : 86 (expected: 86 )
Embedding model   : sentence-transformers/all-MiniLM-L6-v2 (384 dims)
Chunk size/overlap: 800 / 150


---
### 2.4 Retrieval & Prompting

**Goal:** given a question → retrieve the most relevant chunks → build a grounded prompt → call a **local Ollama LLM** → return an answer with **citation-style grounding**.

**Design**

1. **`retrieve()`** embeds the query with the *same* model and returns the top-k chunks with source metadata and a similarity score (`1 − cosine distance`).
2. **Similarity gate (abstention).** If even the best chunk is below `min_similarity`, we skip the LLM and answer *"I don't know based on the provided documents."* This is the cheapest and most reliable defence against hallucination on out-of-scope questions.
3. **Prompt template.** Context excerpts are numbered `[S1]…[Sk]` with their source and page. The system prompt enforces: *use only the context*, *cite every claim as `[S#]`*, and *use an exact refusal sentence if the answer is missing*. Temperature is `0` for reproducibility.
4. **Citation post-processing.** We parse `[S#]` tags from the answer, validate that each refers to a real retrieved chunk, and map them back to `source / page / chunk`. Invalid or missing citations are flagged, not trusted.

In [13]:
def retrieve(query: str, collection, embedder: SentenceTransformer, top_k: int) -> List[Dict]:
    """Embed the query and return the `top_k` nearest chunks with metadata and similarity scores."""
    q_emb = embedder.encode([query], normalize_embeddings=True, convert_to_numpy=True)
    res = collection.query(
        query_embeddings=q_emb.tolist(),
        n_results=top_k,
        include=["documents", "metadatas", "distances"],
    )
    hits = []
    for rank, (cid, text, meta, dist) in enumerate(
        zip(res["ids"][0], res["documents"][0], res["metadatas"][0], res["distances"][0]), start=1
    ):
        hits.append({
            "rank": rank,
            "chunk_id": cid,
            "text": text,
            "source": meta["source"],
            "page": meta["page"],
            "chunk_index": meta["chunk_index"],
            "distance": float(dist),
            "similarity": float(1.0 - dist),       # valid because vectors are normalized + cosine space
        })
    return hits


def format_source(hit: Dict) -> str:
    """Human-readable citation, e.g. 'hx200_manual.txt, chunk 1' or 'report.pdf, p.7, chunk 2'."""
    page = f", p.{hit['page']}" if hit["page"] else ""
    return f"{hit['source']}{page}, chunk {hit['chunk_index']}"


def short_source(hit: Dict) -> str:
    """Compact form used in the evaluation table."""
    page = f"p{hit['page']}" if hit["page"] else ""
    return f"{hit['source']}{('#' + page) if page else ''}#c{hit['chunk_index']} ({hit['similarity']:.2f})"

In [14]:
REFUSAL = "I don't know based on the provided documents."

SYSTEM_PROMPT = """You are a careful assistant that answers questions strictly from the numbered context excerpts provided.

Rules:
1. Use ONLY information found in the context. Never use outside knowledge, and never guess.
2. Cite the excerpt(s) that support every factual statement using square-bracket tags such as [S1] or [S2][S3].
3. If the context does not contain the answer, reply with exactly this sentence and nothing else: I don't know based on the provided documents.
4. Be concise: one to three sentences."""

USER_TEMPLATE = """Context excerpts:
{context}

Question: {question}

Answer (remember the [S#] citations):"""


def build_context(hits: List[Dict]) -> str:
    """Number the retrieved chunks so the model can cite them as [S1], [S2], ..."""
    blocks = []
    for i, h in enumerate(hits, start=1):
        page = f", page {h['page']}" if h["page"] else ""
        blocks.append(f"[S{i}] (source: {h['source']}{page})\n{h['text']}")
    return "\n\n".join(blocks)


def build_user_prompt(question: str, hits: List[Dict]) -> str:
    return USER_TEMPLATE.format(context=build_context(hits), question=question)


# Preview the template with dummy data so the report shows exactly what the LLM sees.
_demo_hits = [{"source": "example.pdf", "page": 3, "chunk_index": 0, "text": "The warranty period is 24 months."}]
print(SYSTEM_PROMPT, "\n" + "-" * 60)
print(build_user_prompt("How long is the warranty?", _demo_hits))

You are a careful assistant that answers questions strictly from the numbered context excerpts provided.

Rules:
1. Use ONLY information found in the context. Never use outside knowledge, and never guess.
2. Cite the excerpt(s) that support every factual statement using square-bracket tags such as [S1] or [S2][S3].
3. If the context does not contain the answer, reply with exactly this sentence and nothing else: I don't know based on the provided documents.
4. Be concise: one to three sentences. 
------------------------------------------------------------
Context excerpts:
[S1] (source: example.pdf, page 3)
The warranty period is 24 months.

Question: How long is the warranty?

Answer (remember the [S#] citations):


In [15]:
def ollama_available_models() -> List[str]:
    """List locally pulled model names (works with both the old dict and new typed client responses)."""
    names = []
    for m in ollama.list()["models"]:
        try:
            names.append(m["model"])
        except (KeyError, TypeError):
            names.append(m["name"])
    return names


def check_ollama(cfg: RAGConfig) -> bool:
    """Preflight: is the server reachable and is the configured model pulled?"""
    try:
        names = ollama_available_models()
    except Exception as exc:
        print(f"⚠️  Cannot reach the Ollama server ({type(exc).__name__}). Start it with `ollama serve`.")
        return False
    if not any(n == cfg.ollama_model or n.startswith(cfg.ollama_model + ":") for n in names):
        print(f"⚠️  Model '{cfg.ollama_model}' is not pulled. Run: ollama pull {cfg.ollama_model}\n   Available: {names}")
        return False
    print(f"✅ Ollama is reachable and '{cfg.ollama_model}' is available.")
    return True


def call_ollama(system_prompt: str, user_prompt: str, cfg: RAGConfig) -> str:
    """Send one grounded chat request to the local model."""
    try:
        resp = ollama.chat(
            model=cfg.ollama_model,
            messages=[
                {"role": "system", "content": system_prompt},
                {"role": "user", "content": user_prompt},
            ],
            options={"temperature": cfg.temperature, "num_ctx": cfg.num_ctx, "seed": 42},
        )
    except ollama.ResponseError as exc:      # server answered with an error (e.g. model not found)
        raise RuntimeError(f"Ollama error {exc.status_code}: {exc.error}. Did you `ollama pull {cfg.ollama_model}`?") from exc
    except Exception as exc:                 # connection refused, timeout, ...
        raise RuntimeError("Could not reach the Ollama server. Start it with `ollama serve`.") from exc
    return resp["message"]["content"].strip()


def parse_citations(answer: str, n_hits: int) -> Tuple[List[int], List[int]]:
    """Extract [S#] tags (also handles '[S1, S2]'). Returns (valid_indices, invalid_indices)."""
    found = set()
    for group in re.findall(r"\[([^\]]*S\d+[^\]]*)\]", answer):
        found.update(int(n) for n in re.findall(r"S(\d+)", group))
    valid = sorted(i for i in found if 1 <= i <= n_hits)
    invalid = sorted(i for i in found if not 1 <= i <= n_hits)
    return valid, invalid


REFUSAL_PATTERN = re.compile(
    r"(don.?t know|do not know|not (?:contain|mention|specif|provide|includ)|no (?:relevant )?information|cannot (?:be )?(?:found|determined))",
    re.IGNORECASE,
)


def is_refusal(answer: str) -> bool:
    return bool(REFUSAL_PATTERN.search(answer))


def answer_question(question: str, collection, embedder: SentenceTransformer, cfg: RAGConfig) -> Dict:
    """Full RAG turn: retrieve -> gate -> prompt -> generate -> validate citations."""
    t0 = time.time()
    hits = retrieve(question, collection, embedder, cfg.top_k)
    top_sim = hits[0]["similarity"] if hits else 0.0

    # Similarity gate: nothing relevant retrieved -> refuse without calling the LLM.
    if not hits or top_sim < cfg.min_similarity:
        answer, gated = REFUSAL, True
    else:
        answer, gated = call_ollama(SYSTEM_PROMPT, build_user_prompt(question, hits), cfg), False

    valid, invalid = parse_citations(answer, len(hits))
    cited_hits = [hits[i - 1] for i in valid]

    # Human-friendly answer: the model's text plus a Sources footer built from validated citations.
    footer = "\n".join(f"  [S{i}] {format_source(hits[i - 1])}" for i in valid)
    display_answer = answer + (f"\n\nSources:\n{footer}" if footer else "")

    return {
        "question": question,
        "answer": answer,                      # raw model answer (used for grading)
        "display_answer": display_answer,      # answer + resolved sources (used for printing / API responses)
        "hits": hits,
        "cited_hits": cited_hits,
        "invalid_citations": invalid,
        "uncited": (not valid) and (not is_refusal(answer)),
        "gated_before_llm": gated,
        "top_similarity": top_sim,
        "retrieved_summary": "; ".join(short_source(h) for h in hits),
        "latency_s": round(time.time() - t0, 2),
    }


OLLAMA_OK = check_ollama(CFG)

✅ Ollama is reachable and 'llama3.2' is available.


**Test set.** 13 questions: **11 answerable** (the answer is in the corpus, with a known source file and a regex that a correct answer must match) and **2 unanswerable** (deliberately absent from the corpus, to test whether the system refuses instead of hallucinating). The questions cover all 10 documents. Some are deliberately harder because the same number or phrase appears in more than one document (for example *1 hour* or *500*), so a retrieval mix-up would produce a plausible but wrongly grounded answer. The project requires at least 10 questions; extend `TEST_QUESTIONS` to add more.

In [16]:
# Each spec: question, whether the corpus contains the answer, the source file that holds it,
# and regex patterns (ALL must match) that a correct answer must contain.
TEST_QUESTIONS: List[Dict] = [
    {"question": "How many days of annual leave do full-time employees get per year?",
     "answerable": True, "expected_source": "hr_leave_policy.pdf", "expected_patterns": [r"\b21\b"]},
    {"question": "What is the maximum nightly hotel cost in Tier-1 cities such as New York, London and Singapore?",
     "answerable": True, "expected_source": "expense_policy.pdf", "expected_patterns": [r"\$?\s?260"]},
    {"question": "How many operating hours are there between full inspections of the HX-200?",
     "answerable": True, "expected_source": "hx200_manual.pdf", "expected_patterns": [r"\b500\b"]},
    {"question": "Within how many hours must a critical vulnerability be patched?",
     "answerable": True, "expected_source": "security_policy.pdf", "expected_patterns": [r"\b72\b"]},
    {"question": "How quickly must the on-call engineer respond to a SEV1 incident?",
     "answerable": True, "expected_source": "incident_runbook.md", "expected_patterns": [r"15\s?min"]},
    {"question": "How long is the standard warranty for the HX-200?",
     "answerable": True, "expected_source": "warranty_and_support.pdf", "expected_patterns": [r"(24[ -]months?|two years|2 years)"]},
    {"question": "What is the list price of one HX-200 robot?",
     "answerable": True, "expected_source": "customer_faq.txt", "expected_patterns": [r"24,?500"]},
    {"question": "What minimum test coverage is required for new code?",
     "answerable": True, "expected_source": "engineering_handbook.md", "expected_patterns": [r"80\s?(%|percent)"]},
    {"question": "How many days per year may an employee work remotely from another country?",
     "answerable": True, "expected_source": "remote_work_policy.txt", "expected_patterns": [r"\b20\b"]},
    {"question": "How long is the probation period for new hires?",
     "answerable": True, "expected_source": "onboarding_guide.txt", "expected_patterns": [r"\b(6|six)[ -]months?"]},
    # Harder: "1 hour" also appears in the incident runbook (SEV2) and the warranty terms (Premium P1 response).
    {"question": "How quickly must a lost or stolen laptop be reported to the security team?",
     "answerable": True, "expected_source": "security_policy.pdf", "expected_patterns": [r"\b(1|one)[ -]hour"]},
    # --- Unanswerable: the correct behaviour is to refuse ---
    {"question": "What is the annual salary of the CEO of Helios Robotics?",
     "answerable": False, "expected_source": None, "expected_patterns": []},
    {"question": "Under which stock ticker symbol is Helios Robotics publicly traded?",
     "answerable": False, "expected_source": None, "expected_patterns": []},
]
assert len(TEST_QUESTIONS) >= 10, "The project requires at least 10 test questions"
print(f"{len(TEST_QUESTIONS)} test questions: "
      f"{sum(q['answerable'] for q in TEST_QUESTIONS)} answerable, {sum(not q['answerable'] for q in TEST_QUESTIONS)} unanswerable")

13 test questions: 11 answerable, 2 unanswerable


In [17]:
records: List[Dict] = []

for i, spec in enumerate(TEST_QUESTIONS, start=1):
    rec = answer_question(spec["question"], collection, embedder, CFG)
    records.append(rec)

    print(f"Q{i}: {rec['question']}")
    print(f"    retrieved: {rec['retrieved_summary']}")
    print("    answer   : " + rec["display_answer"].replace("\n", "\n               "))
    flags = []
    if rec["gated_before_llm"]:          flags.append("gated (no LLM call)")
    if rec["invalid_citations"]:         flags.append(f"invalid citations {rec['invalid_citations']}")
    if rec["uncited"]:                   flags.append("NO citation")
    print(f"    [{rec['latency_s']}s]" + (f"  flags: {', '.join(flags)}" if flags else ""))
    print()

n_gated = sum(r["gated_before_llm"] for r in records)
if n_gated >= max(1, len(records) // 2):
    print(f"⚠️  {n_gated}/{len(records)} questions were refused BEFORE reaching the LLM (best similarity < {CFG.min_similarity}).")
    print("    Ollama was not even called for those. Most likely the questions do not relate to your documents.")
    print("    Fix the questions, or (if they ARE relevant) lower min_similarity in RAGConfig, e.g. 0.15.")

Q1: How many days of annual leave do full-time employees get per year?
    retrieved: hr_leave_policy.pdf#p1#c2 (0.56); hr_leave_policy.pdf#p1#c1 (0.53); hr_leave_policy.pdf#p1#c0 (0.52); hr_leave_policy.pdf#p1#c3 (0.51); hr_leave_policy.pdf#p3#c2 (0.48)
    answer   : Full-time employees accrue 1.75 days of annual leave per month, which equals 21 days per year [S1][S3].
               
               Sources:
                 [S1] hr_leave_policy.pdf, p.1, chunk 2
                 [S3] hr_leave_policy.pdf, p.1, chunk 0
    [15.27s]

Q2: What is the maximum nightly hotel cost in Tier-1 cities such as New York, London and Singapore?
    retrieved: expense_policy.pdf#p1#c1 (0.80); expense_policy.pdf#p1#c2 (0.45); expense_policy.pdf#p3#c0 (0.40); expense_policy.pdf#p1#c3 (0.32); remote_work_policy.txt#c5 (0.30)
    answer   : The maximum nightly hotel cost in Tier-1 cities, such as New York, London, and Singapore, is $260 per night, as stated in [S1] (page 1).
               
            

---
### 2.5 Vision Component

**Not applicable.** This project follows the **Core Track** (text-only RAG); the Extended Track's computer-vision/YOLO component is not implemented.

---
### 2.6 Evaluation

**Goal:** a transparent results table for the test questions.

**Grading rubric (automatic first pass, manual override available)**

| Label | Meaning |
|---|---|
| **Correct** | *Answerable question:* the answer matches the expected facts **and** cites a valid chunk from the expected source. *Unanswerable question:* the system correctly refuses. |
| **Hallucinated** | The answer is wrong, ungrounded (no valid citation to the right source), cites a non-existent excerpt, or answers a question the corpus cannot answer. |
| **Missed (false refusal)** | The information *was* in the corpus but the system said it didn't know. This is a recall failure, not a fabrication, so it is kept separate. |

The main table has exactly the four requested columns. Extra diagnostics (similarity, whether the expected source was retrieved, the grading reason) live in a second DataFrame. Automatic grading is a heuristic; **read every answer** and use `MANUAL_OVERRIDES` where you disagree.

In [18]:
MANUAL_OVERRIDES: Dict[int, str] = {}      # e.g. {3: "Correct"} -> question numbers are 1-based


def grade_answer(rec: Dict, spec: Dict) -> Tuple[str, str]:
    """Return (label, reason) following the rubric above."""
    refused = is_refusal(rec["answer"])

    # Unanswerable: the only acceptable behaviour is to refuse.
    if not spec["answerable"]:
        return ("Correct", "correctly refused") if refused else ("Hallucinated", "answered an unanswerable question")

    if refused:
        in_retrieved = spec["expected_source"] in {h["source"] for h in rec["hits"]}
        return "Missed (false refusal)", ("expected source was retrieved but LLM refused"
                                          if in_retrieved else "expected source was not retrieved")

    facts_ok = all(re.search(p, rec["answer"], re.IGNORECASE) for p in spec["expected_patterns"])
    cited_sources = {h["source"] for h in rec["cited_hits"]}
    grounded = spec["expected_source"] in cited_sources

    if rec["invalid_citations"]:
        return "Hallucinated", f"cites non-existent excerpt(s) {rec['invalid_citations']}"
    if not facts_ok:
        return "Hallucinated", "expected facts missing or wrong"
    if not grounded:
        return "Hallucinated", "correct-looking answer but not grounded in the expected source (missing/wrong citation)"
    return "Correct", "facts match and citation points to the expected source"


rows, diag = [], []
for i, (spec, rec) in enumerate(zip(TEST_QUESTIONS, records), start=1):
    label, reason = grade_answer(rec, spec)
    if i in MANUAL_OVERRIDES:
        label, reason = MANUAL_OVERRIDES[i], reason + " (manual override)"

    rows.append({
        "Question": rec["question"],
        "Retrieved Source": rec["retrieved_summary"],
        "Answer": rec["answer"],
        "Correct/Hallucinated": label,
    })
    diag.append({
        "Q#": i,
        "answerable": spec["answerable"],
        "top_similarity": round(rec["top_similarity"], 3),
        "expected_source_retrieved": (spec["expected_source"] in {h["source"] for h in rec["hits"]}) if spec["answerable"] else None,
        "gated_before_llm": rec["gated_before_llm"],
        "label": label,
        "reason": reason,
        "latency_s": rec["latency_s"],
    })

results_df = pd.DataFrame(rows, columns=["Question", "Retrieved Source", "Answer", "Correct/Hallucinated"])
results_df.index = pd.RangeIndex(1, len(results_df) + 1, name="Q#")
diagnostics_df = pd.DataFrame(diag).set_index("Q#")

with pd.option_context("display.max_colwidth", 160, "display.width", 250):
    display(results_df)

,Question,Retrieved Source,Answer,Correct/Hallucinated
Q#,,,,
1,How many days of annual leave do full-time employees get per year?,hr_leave_policy.pdf#p1#c2 (0.56); hr_leave_policy.pdf#p1#c1 (0.53); hr_leave_policy.pdf#p1#c0 (0.52); hr_leave_policy.pdf#p1#c3 (0.51); hr_leave_policy.pdf#...,"Full-time employees accrue 1.75 days of annual leave per month, which equals 21 days per year [S1][S3].",Correct
2,"What is the maximum nightly hotel cost in Tier-1 cities such as New York, London and Singapore?",expense_policy.pdf#p1#c1 (0.80); expense_policy.pdf#p1#c2 (0.45); expense_policy.pdf#p3#c0 (0.40); expense_policy.pdf#p1#c3 (0.32); remote_work_policy.txt#c...,"The maximum nightly hotel cost in Tier-1 cities, such as New York, London, and Singapore, is $260 per night, as stated in [S1] (page 1).",Correct
3,How many operating hours are there between full inspections of the HX-200?,engineering_handbook.md#c4 (0.54); warranty_and_support.pdf#p2#c2 (0.52); warranty_and_support.pdf#p3#c0 (0.52); onboarding_guide.txt#c2 (0.51); customer_fa...,I don't know based on the provided documents.,Missed (false refusal)
4,Within how many hours must a critical vulnerability be patched?,security_policy.pdf#p3#c0 (0.55); security_policy.pdf#p3#c1 (0.50); engineering_handbook.md#c5 (0.46); incident_runbook.md#c6 (0.43); incident_runbook.md#c7...,"According to [S1], critical vulnerabilities must be patched within 72 hours.",Correct
5,How quickly must the on-call engineer respond to a SEV1 incident?,incident_runbook.md#c6 (0.73); incident_runbook.md#c0 (0.68); incident_runbook.md#c7 (0.62); incident_runbook.md#c2 (0.61); incident_runbook.md#c4 (0.55),Within 15 minutes; acknowledge the page within 5 minutes [S1].,Correct
6,How long is the standard warranty for the HX-200?,warranty_and_support.pdf#p3#c0 (0.67); warranty_and_support.pdf#p1#c0 (0.60); warranty_and_support.pdf#p2#c1 (0.57); customer_faq.txt#c5 (0.54); warranty_an...,"The standard warranty for the HX-200 is 24 months from the delivery date, as stated in [S1] and [S2].",Correct
7,What is the list price of one HX-200 robot?,customer_faq.txt#c1 (0.67); hx200_manual.pdf#p1#c1 (0.58); hx200_manual.pdf#p1#c0 (0.55); customer_faq.txt#c3 (0.54); warranty_and_support.pdf#p1#c0 (0.52),"The list price of one HX-200 robot is $24,500 per robot, excluding taxes, freight, and optional accessories. [S1]",Correct
8,What minimum test coverage is required for new code?,engineering_handbook.md#c7 (0.57); engineering_handbook.md#c6 (0.39); engineering_handbook.md#c2 (0.38); engineering_handbook.md#c3 (0.33); engineering_hand...,The minimum test coverage required for new code is 80%. [S1][S2],Correct
9,How many days per year may an employee work remotely from another country?,remote_work_policy.txt#c2 (0.63); remote_work_policy.txt#c3 (0.54); remote_work_policy.txt#c0 (0.49); remote_work_policy.txt#c4 (0.49); hr_leave_policy.pdf#...,"According to [S1], an employee may work remotely from another country for up to 20 days per calendar year.",Correct


In [19]:
# ---- Aggregate metrics ----
counts = results_df["Correct/Hallucinated"].value_counts()
n = len(results_df)
answerable = diagnostics_df[diagnostics_df["answerable"]]
unanswerable = diagnostics_df[~diagnostics_df["answerable"]]

eval_summary = {
    "n_questions": n,
    "correct": int(counts.get("Correct", 0)),
    "hallucinated": int(counts.get("Hallucinated", 0)),
    "missed_false_refusal": int(counts.get("Missed (false refusal)", 0)),
    "accuracy": round(counts.get("Correct", 0) / n, 3),
    "retrieval_hit_rate_answerable": round(float(answerable["expected_source_retrieved"].mean()), 3),
    "correct_refusals_unanswerable": f"{int((unanswerable['label'] == 'Correct').sum())}/{len(unanswerable)}",
    "median_latency_s": float(diagnostics_df["latency_s"].median()),
}
print(json.dumps(eval_summary, indent=2))

# ---- Observed failures in THIS run (fully generated from the data above) ----
failed = diagnostics_df[diagnostics_df["label"] != "Correct"]
if failed.empty:
    print("\nNo failures observed in this run on the test questions.")
else:
    print(f"\n{len(failed)} failure(s) observed in this run:")
    for qn, row in failed.iterrows():
        print(f"  Q{qn} [{row['label']}] top_sim={row['top_similarity']}  ->  {row['reason']}")
        print(f"       Q: {results_df.loc[qn, 'Question']}")
        print(f"       A: {results_df.loc[qn, 'Answer'][:200]}")

{
  "n_questions": 13,
  "correct": 12,
  "hallucinated": 0,
  "missed_false_refusal": 1,
  "accuracy": 0.923,
  "retrieval_hit_rate_answerable": 0.909,
  "correct_refusals_unanswerable": "2/2",
  "median_latency_s": 4.03
}

1 failure(s) observed in this run:
  Q3 [Missed (false refusal)] top_sim=0.538  ->  expected source was not retrieved
       Q: How many operating hours are there between full inspections of the HX-200?
       A: I don't know based on the provided documents.


#### 2.6 Failure cases and mitigation strategies

**Observed results.** On the 13 test questions the pipeline gave 12 correct answers (92.3%), no hallucinations, and refused both out-of-scope questions (the CEO's salary and the stock ticker) instead of inventing an answer. Median latency was 2.9 s per question with a local `llama3.2` model.

**Failure case (Q3).** The question *"How many operating hours are there between full inspections of the HX-200?"* was refused (*false refusal*). The retrieval step returned chunks from `engineering_handbook.md`, `warranty_and_support.pdf`, `onboarding_guide.txt` and `customer_faq.txt` (similarity around 0.5), but **not** the `hx200_manual.pdf` chunk that contains the 500-hour inspection interval, so the model had no evidence and correctly said it did not know. The failure is therefore a **retrieval miss**, not a generation error: the words of the question (*HX-200*, *inspection*, *hours*) also occur in several other documents (warranty terms, FAQ, handbook), and the general-purpose embedding model ranked those overview passages above the specific maintenance paragraph. Because the safety mechanisms worked (strict prompt, exact refusal sentence), the system failed *safely*: it withheld an answer instead of guessing.

**A weaker point (Q11).** The question about reporting a lost laptop was graded correct (it states the 1-hour deadline and cites `security_policy.pdf`), but the answer opened with an unrelated sentence about reporting incidents in good faith. The chunk mixes several rules, and the small model repeated neighbouring text. The answer is right but less focused than it should be.

**Mitigations already in place:** temperature 0 and a fixed seed; a system prompt that allows only the retrieved context and defines an exact refusal sentence; a similarity threshold that skips the LLM when retrieval is weak; whitespace-aware chunking with overlap; validation of `[S#]` citations; and grading that requires the citation to point to the expected source, so a lucky guess is not counted as correct. **Next steps:** (a) increase `top_k` from 5 to 8 so that lower-ranked but relevant chunks reach the model; (b) hybrid retrieval (BM25 keyword search plus embeddings) so exact terms such as *inspection* and *HX-200* count more; (c) a cross-encoder reranker over the top 20 candidates; (d) section-based chunking (split at headings) so each chunk covers one rule; (e) tell the model to answer only the question asked; and (f) a larger labelled question set, since 13 questions is a smoke test rather than a benchmark.


---
### 2.7 Export

**Goal:** leave `data/vector_store/` in a state where a **FastAPI backend can start, load the index, and answer queries — without this notebook and without re-embedding.**

**Contents of `data/vector_store/`**

| Path | Purpose |
|---|---|
| `chroma/` | The persisted ChromaDB database (vectors, texts, metadata, HNSW index) |
| `config.json` | Chunk size & overlap, **embedding model name**, embedding dimension, collection name, distance metric, `top_k`, similarity threshold, Ollama model, prompt templates, corpus fingerprint, build time, evaluation summary |

The backend must load **the embedding model named in `config.json`**. Using a different model at query time would produce vectors from a different space and silently return garbage.

In [20]:
# Enrich config.json with everything the backend needs to reproduce the RAG turn exactly.
final_cfg = save_store_config(CFG, {
    "prompt": {
        "system_prompt": SYSTEM_PROMPT,
        "user_template": USER_TEMPLATE,
        "refusal_sentence": REFUSAL,
        "citation_format": "[S#] -> numbered context excerpts",
    },
    "evaluation": eval_summary,
    "notebook": "rag_pipeline.ipynb",
    "exported_at_utc": datetime.now(timezone.utc).isoformat(timespec="seconds"),
})

# Keep the evaluation table alongside the data for reference (not required by the backend).
Path("D:/ITI Computer Vision/Final Project/data").mkdir(exist_ok=True)
results_df.to_csv("D:/ITI Computer Vision/Final Project/data/evaluation_results.csv")

# ---- Show what was exported ----
vs = Path(CFG.vector_store_dir)
print(f"Contents of {vs}/")
for p in sorted(vs.rglob("*")):
    if p.is_file():
        print(f"  {p.relative_to(vs).as_posix():<45}{p.stat().st_size / 1024:>10.1f} KB")

print("\nKey settings saved in config.json:")
for k in ["chunk_size", "chunk_overlap", "embedding_model", "embedding_dimension",
          "collection_name", "distance_metric", "num_chunks", "top_k", "min_similarity", "ollama_model"]:
    print(f"  {k:<20}{final_cfg[k]}")

Contents of D:\ITI Computer Vision\Final Project\data\vector_store/
  chroma/b8ddd290-1840-4971-bf6d-82da8b8ebb61/data_level0.bin     163.7 KB
  chroma/b8ddd290-1840-4971-bf6d-82da8b8ebb61/header.bin       0.1 KB
  chroma/b8ddd290-1840-4971-bf6d-82da8b8ebb61/length.bin       0.4 KB
  chroma/b8ddd290-1840-4971-bf6d-82da8b8ebb61/link_lists.bin       0.0 KB
  chroma/chroma.sqlite3                            1128.0 KB
  config.json                                         1.9 KB

Key settings saved in config.json:
  chunk_size          800
  chunk_overlap       150
  embedding_model     sentence-transformers/all-MiniLM-L6-v2
  embedding_dimension 384
  collection_name     rag_chunks
  distance_metric     cosine
  num_chunks          86
  top_k               5
  min_similarity      0.25
  ollama_model        llama3.2


In [21]:
# ---- Backend-readiness test: load the store in a BRAND-NEW Python process (no notebook state). ----
loader_script = """
import json, sys
from pathlib import Path
import chromadb
from sentence_transformers import SentenceTransformer

vs = Path(sys.argv[1])
cfg = json.loads((vs / "config.json").read_text(encoding="utf-8"))
collection = chromadb.PersistentClient(path=str(vs / "chroma")).get_collection(cfg["collection_name"])
model = SentenceTransformer(cfg["embedding_model"])

q = model.encode(["How many days of annual leave do employees get?"], normalize_embeddings=True)
res = collection.query(query_embeddings=q.tolist(), n_results=1, include=["metadatas", "distances"])
assert collection.count() == cfg["num_chunks"], "chunk count mismatch"
assert len(q[0]) == cfg["embedding_dimension"], "embedding dimension mismatch"
print("OK | chunks:", collection.count(), "| top hit:", res["metadatas"][0][0]["source"], "| sim:", round(1 - res["distances"][0][0], 3))
"""

proc = subprocess.run([sys.executable, "-c", loader_script, str(vs)], capture_output=True, text=True)
print(proc.stdout.strip() or proc.stderr.strip()[-800:])
assert proc.returncode == 0, "Fresh-process load failed: the exported store is not backend-ready"

OK | chunks: 86 | top hit: hr_leave_policy.pdf | sim: 0.634


#### Loading the store from FastAPI (reference sketch)

```python
# app/main.py — run with:  uvicorn app.main:app --reload
import json
from contextlib import asynccontextmanager
from pathlib import Path

import chromadb
from fastapi import FastAPI
from pydantic import BaseModel
from sentence_transformers import SentenceTransformer

VS_DIR = Path("data/vector_store")
state = {}

@asynccontextmanager
async def lifespan(app: FastAPI):
    cfg = json.loads((VS_DIR / "config.json").read_text(encoding="utf-8"))
    client = chromadb.PersistentClient(path=str(VS_DIR / "chroma"))
    state["cfg"] = cfg
    state["collection"] = client.get_collection(cfg["collection_name"])
    state["embedder"] = SentenceTransformer(cfg["embedding_model"])   # SAME model as at index time
    yield
    state.clear()

app = FastAPI(lifespan=lifespan)

class Query(BaseModel):
    question: str

@app.post("/ask")
def ask(q: Query):
    # 1) embed the question   2) collection.query(top_k=state["cfg"]["top_k"])
    # 3) build the prompt from cfg["prompt"]   4) ollama.chat(model=cfg["ollama_model"], ...)
    # 5) return the answer plus the validated [S#] sources
    ...
```

In production, wrap the retrieval/generation logic from section 2.4 into a shared module (`rag_core.py`) that both this notebook and the API import, so the two can never drift apart.

---
## Conclusion & Next Steps

The pipeline **loads → chunks → embeds → persists → retrieves → generates with citations → evaluates → exports**. The index is persisted in `data/vector_store/`, described by `config.json`, and verified to load in a fresh process.

**Recommended follow-ups:** (1) index your real corpus and replace `TEST_QUESTIONS` with 30–50 labelled questions; (2) add hybrid retrieval and a reranker if the retrieval hit rate is below target; (3) add a faithfulness check; (4) move the core functions into a module and expose them through FastAPI; (5) add incremental indexing (only re-embed changed files) once the corpus grows.